In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import statsmodels.api as sm
from sklearn.preprocessing import MultiLabelBinarizer

processed_dir = Path("../data/processed")

# Load the datasets
consumption_analysis = pd.read_parquet(processed_dir / "consumption_analysis.parquet")
recommendations_clean = pd.read_parquet(processed_dir / "recommendations_clean.parquet")
ratings_clean = pd.read_parquet(processed_dir / "ratings_clean.parquet")
movies_clean = pd.read_parquet(processed_dir / "movies_clean.parquet")
model_df = consumption_analysis.copy()
model_df = model_df.rename(columns={"consumed_after_recommendation": "consumed"})

## Q3: Among recommended movies, which factors are associated with subsequent consumption?

We model whether a recommended movie is subsequently consumed, where consumption is proxied by a later user rating. Because this outcome is binary, we use logistic regression. The main predictor is `predictedRating`; controls capture information available at recommendation time, including prior movie popularity, prior user activity, genre, prior exposure to the movie, and the amount of observable follow-up time. Results should be interpreted as associations among recommendations, not as causal effects of recommendation exposure.

Controls:

| Feature                  | Rationale                                                                                                                          |
| ------------------------ | ---------------------------------------------------------------------------------------------------------------------------------- |
| Movie popularity         | Popular movies may be consumed regardless of the recommendation.                                                                   |
| User activity            | Users who rate more often may be more likely to consume and rate recommended movies.                                               |
| Genre                    | Consumption rates may differ across movie genres.                                                                                  |
| Attribution window       | Recommendations with more time before the next recommendation have more opportunity to be followed by a rating. |
| Previous recommendations | Repeated recommendations of the same movie may affect consumption differently than a first recommendation.                         |

In [2]:
model_df.head()

,user_id,movie_id,recommendation_tstamp,predictedRating,recommendation_id,belief_tstamp,user_predict_rating,rating_tstamp,rating,time_to_consumption,consumed,prior_rating_tstamp,prior_rating,prior_consumption
0,377084,924,2023-03-01 06:10:51,4.303385,2,NaT,NaN,NaT,NaN,NaT,False,NaT,NaN,False
1,377084,1201,2023-03-01 06:10:51,4.215998,5,NaT,NaN,NaT,NaN,NaT,False,NaT,NaN,False
2,377084,1204,2023-03-01 06:10:51,4.236355,7,NaT,NaN,NaT,NaN,NaT,False,NaT,NaN,False
3,377084,1258,2023-03-01 06:10:51,4.241269,4,NaT,NaN,NaT,NaN,NaT,False,NaT,NaN,False
4,377084,1732,2023-03-01 06:10:51,4.255416,3,NaT,NaN,NaT,NaN,NaT,False,NaT,NaN,False


In [3]:
model_df["consumed"] = model_df["consumed"].astype(int)  # Convert consumed to int for logistic regression

In [4]:
model_df = model_df[["user_id", "movie_id", "predictedRating", "recommendation_tstamp", "consumed"]]

In [5]:
model_df.head()

,user_id,movie_id,predictedRating,recommendation_tstamp,consumed
0,377084,924,4.303385,2023-03-01 06:10:51,0
1,377084,1201,4.215998,2023-03-01 06:10:51,0
2,377084,1204,4.236355,2023-03-01 06:10:51,0
3,377084,1258,4.241269,2023-03-01 06:10:51,0
4,377084,1732,4.255416,2023-03-01 06:10:51,0


## Feature Engineering

In [6]:
# Ensure timestamps have the same precision
model_df["recommendation_tstamp"] = pd.to_datetime(
    model_df["recommendation_tstamp"]
).astype("datetime64[ns]")

ratings_clean["tstamp"] = pd.to_datetime(
    ratings_clean["tstamp"]
).astype("datetime64[ns]")

In [7]:
# Feature: movie popularity
# Rolling count of ratings for each movie up to the recommendation timestamp

ratings_valid = (
    ratings_clean[ratings_clean["rating"] >= 0]
    .sort_values(["movie_id", "tstamp"])
)

ratings_valid["movie_popularity_cumulative"] = (
    ratings_valid.groupby("movie_id").cumcount() + 1
)

model_df = model_df.sort_values("recommendation_tstamp")

model_df = pd.merge_asof(
    model_df,
    ratings_valid[
        ["movie_id", "tstamp", "movie_popularity_cumulative"]
    ].sort_values("tstamp"),
    left_on="recommendation_tstamp",
    right_on="tstamp",
    by="movie_id",
    direction="backward",
    allow_exact_matches=False
)

model_df["movie_popularity_cumulative"] = (
    model_df["movie_popularity_cumulative"].fillna(0)
)

model_df["movie_popularity"] = np.log1p(
    model_df["movie_popularity_cumulative"]
)

In [8]:
# Feature: user activity
# Rolling count of ratings for each user up to the recommendation timestamp

ratings_valid = (
    ratings_clean[ratings_clean["rating"] >= 0]
    .sort_values(["user_id", "tstamp"])
)

ratings_valid["user_activity_cumulative"] = (
    ratings_valid.groupby("user_id").cumcount() + 1
)

model_df = model_df.sort_values("recommendation_tstamp")

model_df = pd.merge_asof(
    model_df,
    ratings_valid[
        ["user_id", "tstamp", "user_activity_cumulative"]
    ].sort_values("tstamp"),
    left_on="recommendation_tstamp",
    right_on="tstamp",
    by="user_id",
    direction="backward",
    allow_exact_matches=False
)

model_df["user_activity_cumulative"] = (
    model_df["user_activity_cumulative"].fillna(0)
)

model_df["user_activity"] = np.log1p(
    model_df["user_activity_cumulative"]
)

In [9]:
# Drop the temporary columns
model_df = model_df.drop(columns=["tstamp_x", "tstamp_y", "movie_popularity_cumulative", "user_activity_cumulative"])

In [10]:
model_df.head()

,user_id,movie_id,predictedRating,recommendation_tstamp,consumed,movie_popularity,user_activity
0,377084,924,4.303385,2023-03-01 06:10:51,0,6.748760,6.647688
1,377084,1201,4.215998,2023-03-01 06:10:51,0,6.371612,6.647688
2,377084,1204,4.236355,2023-03-01 06:10:51,0,5.765191,6.647688
3,377084,1258,4.241269,2023-03-01 06:10:51,0,6.872128,6.647688
4,377084,1732,4.255416,2023-03-01 06:10:51,0,6.893656,6.647688


In [11]:
# Feature: movie genres
movies_clean.head()

mlb = MultiLabelBinarizer()

genre_binary = pd.DataFrame(
    mlb.fit_transform(movies_clean["genres"]),
    columns=mlb.classes_,
    index=movies_clean.index
)

genre_binary["movie_id"] = movies_clean["movie_id"].values

In [12]:
# Keep only genres with at least 1,000 movies
MIN_MOVIES = 1000  # threshold

genre_totals = genre_binary.drop(columns="movie_id").sum()
keep_genres = genre_totals[genre_totals >= MIN_MOVIES].index.tolist()
genre_binary = genre_binary[["movie_id"] + keep_genres]

In [13]:
# Merge the binary genre features into the model_df
model_df = model_df.merge(
    genre_binary,
    on="movie_id",
    how="left"
)

In [14]:
model_df.head()

,user_id,movie_id,predictedRating,recommendation_tstamp,consumed,movie_popularity,user_activity,Action,Adventure,Animation,...,Fantasy,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,Unknown,War,Western
0,377084,924,4.303385,2023-03-01 06:10:51,0,6.748760,6.647688,0,1,0,...,0,0,0,0,0,1,0,0,0,0
1,377084,1201,4.215998,2023-03-01 06:10:51,0,6.371612,6.647688,1,1,0,...,0,0,0,0,0,0,0,0,0,1
2,377084,1204,4.236355,2023-03-01 06:10:51,0,5.765191,6.647688,0,1,0,...,0,0,0,0,0,0,0,0,1,0
3,377084,1258,4.241269,2023-03-01 06:10:51,0,6.872128,6.647688,0,0,0,...,0,1,0,0,0,0,0,0,0,0
4,377084,1732,4.255416,2023-03-01 06:10:51,0,6.893656,6.647688,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [15]:
# Feature: attribution window for consumption
# The difference between current recommendation timestamp and next timestamp for same user-movie pairs
# Rationale: After a user receives the same recommendation again, a later rating cannot reliably be
# attributed to the earlier recommendation. Therefore, for a given recommendation,
# we limit the attribution window to the next recommendation of that same movie to the same user.

# Get the next recommendation timestamp for each user-movie pair
rec_sorted = recommendations_clean.sort_values(["user_id", "movie_id", "tstamp"])
rec_sorted["next_recommendation_tstamp"] = (
    rec_sorted.groupby(["user_id", "movie_id"])["tstamp"].shift(-1)
)

rec_sorted = rec_sorted.rename(columns={"tstamp": "recommendation_tstamp"})

# Merge the next recommendation timestamp into the model_df
model_df = model_df.merge(
    rec_sorted[["user_id", "movie_id", "recommendation_tstamp", "next_recommendation_tstamp"]],
    on=["user_id", "movie_id", "recommendation_tstamp"],
    how="left"
)

dataset_end = ratings_clean["tstamp"].max()  # Fall back to the end of the dataset if there is no next recommendation timestamp

window_end = model_df["next_recommendation_tstamp"].fillna(dataset_end)  # Fill missing values with the end of the dataset
model_df["follow_up_days"] = (window_end - model_df["recommendation_tstamp"]).dt.total_seconds() / 86400  # in days
model_df = model_df.drop(columns=["next_recommendation_tstamp"])

In [16]:
# Feature: previous recommendations (number)

# Get the rolling count of recommendations for each user-movie pair
rec_sorted["previous_recommendations"] = (
    rec_sorted.groupby(["user_id", "movie_id"]).cumcount()
)

model_df = model_df.merge(
    rec_sorted[["user_id", "movie_id", "recommendation_tstamp", "previous_recommendations"]],
    on=["user_id", "movie_id", "recommendation_tstamp"],
    how="left"
)

In [17]:
model_df.head()

,user_id,movie_id,predictedRating,recommendation_tstamp,consumed,movie_popularity,user_activity,Action,Adventure,Animation,...,Musical,Mystery,Romance,Sci-Fi,Thriller,Unknown,War,Western,follow_up_days,previous_recommendations
0,377084,924,4.303385,2023-03-01 06:10:51,0,6.748760,6.647688,0,1,0,...,0,0,0,1,0,0,0,0,1.779583,0
1,377084,1201,4.215998,2023-03-01 06:10:51,0,6.371612,6.647688,1,1,0,...,0,0,0,0,0,0,0,1,1.779583,0
2,377084,1204,4.236355,2023-03-01 06:10:51,0,5.765191,6.647688,0,1,0,...,0,0,0,0,0,0,1,0,83.834653,0
3,377084,1258,4.241269,2023-03-01 06:10:51,0,6.872128,6.647688,0,0,0,...,0,0,0,0,0,0,0,0,1.779583,0
4,377084,1732,4.255416,2023-03-01 06:10:51,0,6.893656,6.647688,0,0,0,...,0,0,0,0,0,0,0,0,1.779583,0


In [18]:
# Null counts for follow_up_days and previous_recommendations
null_counts = model_df[["follow_up_days", "previous_recommendations"]].isnull().sum()
print(f"Null counts for follow_up_days and previous_recommendations:\n{null_counts}")

# Null counts for the genre features
print(f"Null counts for genre features:\n{model_df[keep_genres].isnull().sum()}")

Null counts for follow_up_days and previous_recommendations:
follow_up_days              0
previous_recommendations    0
dtype: int64
Null counts for genre features:
Action         0
Adventure      0
Animation      0
Children       0
Comedy         0
Crime          0
Documentary    0
Drama          0
Fantasy        0
Horror         0
Musical        0
Mystery        0
Romance        0
Sci-Fi         0
Thriller       0
Unknown        0
War            0
Western        0
dtype: int64


In [19]:
# Prepare the feature matrix X and target vector y
X = pd.concat([
    model_df[["predictedRating", "movie_popularity", "user_activity",
              "follow_up_days", "previous_recommendations"]],
    model_df[keep_genres]
], axis=1)

X = sm.add_constant(X)  # Add a constant term for the intercept
X = X.astype(float)  # Ensure all features are numeric

y = model_df["consumed"]

In [20]:
logit_model = sm.Logit(y, X).fit()  # Fit the logistic regression model

Optimization terminated successfully.
         Current function value: 0.029951
         Iterations 11


In [21]:
print(logit_model.summary())

                           Logit Regression Results                           
Dep. Variable:               consumed   No. Observations:              1209225
Model:                          Logit   Df Residuals:                  1209201
Method:                           MLE   Df Model:                           23
Date:                Sat, 15 Aug 2026   Pseudo R-squ.:                  0.1004
Time:                        16:05:09   Log-Likelihood:                -36217.
converged:                       True   LL-Null:                       -40258.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
const                       -5.7927      0.188    -30.731      0.000      -6.162      -5.423
predictedRating             -0.0921      0.031     -3.001      0.003      -0.152      -0.

In [22]:
np.exp(logit_model.params)  # Calculate the odds ratios for the coefficients

const                       0.003050
predictedRating             0.911990
movie_popularity            1.175203
user_activity               0.972249
follow_up_days              1.007407
previous_recommendations    0.991000
Action                      1.026077
Adventure                   0.885798
Animation                   1.037955
Children                    1.095843
Comedy                      1.206021
Crime                       0.940990
Documentary                 0.576209
Drama                       1.011153
Fantasy                     1.152973
Horror                      1.186360
Musical                     1.093539
Mystery                     0.950133
Romance                     0.812867
Sci-Fi                      1.319905
Thriller                    1.121260
Unknown                     0.613281
War                         0.690217
Western                     0.928811
dtype: float64

### Odds Ratio (OR) Interpretation

1. **OR = 1** → no change in odds
2. **OR > 1** → higher odds of consumption
3. **OR < 1** → lower odds of consumption

### Key Finding
1. **Predicted rating:** Higher predicted ratings were associated with slightly lower odds of consumption (OR = 0.912, p = 0.003).

2. **Movie popularity:** More popular movies had higher odds of consumption (OR = 1.175, p < 0.001).

3. **Follow-up time:** More follow-up time was associated with higher odds of consumption (OR = 1.007/day, p < 0.001).

4. **Repeat recommendations:** More previous recommendations were associated with slightly lower odds of consumption (OR = 0.991, p < 0.001).

5. **Genres:**
    - Sci-Fi, Comedy, Horror, and Fantasy had **higher odds** of consumption.
    - Documentary, War, and Romance had **lower odds** of consumption.